# Quick Start: Arbitration Executor

This notebook demonstrates how to use the executor framework to analyze ABIDES simulation results.

## 1. Single File Analysis

In [ ]:
from executor import ArbitrationExecutor
from pathlib import Path

# Initialize executor
executor = ArbitrationExecutor(output_dir="../research/output")

# Run analysis on a single results file
report = executor.run_full_analysis(
    results_path="../DeepMarket/ABIDES/research/data/limit_order_qty_0.1.parquet",
    trade_value_usd=100000.0,  # $100k notional
    fee_hurdle_bps=4.0         # 4 bps minimum spread
)

## 2. Access Research Findings

In [ ]:
# RQ1: Decay rate and half-life
rq1 = report['research_questions']['RQ1']
print("Decay Analysis:")
print(f"  Median TTF: {rq1['findings']['time_to_fill_median_ms']:.1f} ms")
print(f"  Mean TTF: {rq1['findings']['time_to_fill_mean_ms']:.1f} ms")

In [ ]:
# RQ2: Fee hurdle thresholds
rq2 = report['research_questions']['RQ2']
print("\nFee Hurdles:")
print(f"  Perp-Spot: {rq2['findings']['perp_spot_same_exchange_bps']:.2f} bps")
print(f"  CEX-to-CEX: {rq2['findings']['cex_to_cex_bps']:.2f} bps")
print(f"  CEX-to-DEX: {rq2['findings']['cex_to_dex_bps']:.2f} bps")

In [ ]:
# RQ3: Capital capacity limits
rq3 = report['research_questions']['RQ3']
print("\nCapital Capacity:")
print(f"  Max profitable size: {rq3['findings']['max_profitable_size_btc']:.2f} BTC")

## 3. Batch Analysis

In [ ]:
from pathlib import Path
import glob

# Find all result files
results_dir = Path("../DeepMarket/ABIDES/research/data")
parquet_files = sorted(results_dir.glob("limit_order_qty_*.parquet"))

print(f"Found {len(parquet_files)} result files")
for pf in parquet_files:
    print(f"  - {pf.name}")

## 4. Custom Analysis

In [ ]:
# You can also use individual modules
from fee_hurdle import FeeHurdleCalculator
from decay_analyzer import DecayAnalyzer
from liquidity_capacity import generate_liquidity_report
import pandas as pd

# Load results manually
df = pd.read_parquet("../DeepMarket/ABIDES/research/data/limit_order_qty_0.1.parquet")

# Custom analysis
fee_calc = FeeHurdleCalculator()
hurdle = fee_calc.calculate_perp_spot_fee_hurdle(
    perp_venue='bybit',
    spot_venue='bybit',
    trade_value_usd=100000.0,
    hold_hours=1.0
)
print(f"Fee hurdle: {hurdle['total_hurdle_bps']:.2f} bps")

## 5. Generate Report

In [ ]:
# Use CLI to generate Markdown report
!python -m executor.cli report \
    ../DeepMarket/ABIDES/research/data/limit_order_qty_0.1.parquet \
    --output ../research/output/report.md

## Next Steps

1. Run the full simulation: `python run_all_qtys.py`
2. Analyze all results: `python -m executor.cli batch data/`
3. Review generated reports in `simulator/research/output/`
4. Use results for your research paper